In [ ]:
import numpy as np
import yfinance as yf
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
from scipy import stats
from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox

from GBM import *

In [ ]:
my_picks = ['AAPL', # Apple Inc. - Information Technology,
            'AMZN', # Amazon - Consumer Discretionary
            'F', # Ford Motor Company - Consumer Discretionary
            'KO', # Coca-Cola Company (The) - Consumer Staples
            'XOM', # ExxonMobil - Energy
            'JPM', # JPMorgan Chase - Financials,
            'PFE', #Pfizer - Health Care,
            'BA', # Boeing - Industrials
            'DD', # DuPont - Materials
            'NRG'] # NRG Energy - Utilities

In [ ]:
# # Either download and save the equity data
# dataSave = yf.download(my_picks, start="2015-01-01", end="2026-01-01",auto_adjust=True)
# dataSave.to_parquet('yfinance_selected.parquet')

# # Or read a previously saved copy
dataSave = pd.read_parquet('yfinance_selected.parquet')

close = dataSave['Close'].copy()
close

In [ ]:
# Fit normal distribution to log increments of price data. Run GBM monte-carlo assuming normal distribution
month_starts = make_month_grid(close)
path_containment = []
standardized_returns = []

for ticker in my_picks:
    
    tic = dt.datetime.today()
    print(ticker,tic,end=' - ')
    
    ticker_standardized_returns = []
    # Keep 3 months for coefficient estimation, also throw out 1 month for last upper bound
    for month_select in range(3,len(month_starts)-1):

        prev_data, window = isolate_data(close,ticker,month_starts,month_select)
        coefficients = estimate_normal_coefficients(prev_data)
        standardized_return = standardize_with_normal_coefficients(window,coefficients)
        ticker_standardized_returns.append(standardized_return)

        ticker_id = my_picks.index(ticker)  # or any stable integer mapping
        time_start = month_starts[month_select]
        ss = np.random.SeedSequence([ticker_id, time_start.toordinal()])
        rng = np.random.default_rng(ss)
        drift, bands, paths = simulate_paths(window,coefficients,rng)

        percentile_dict = make_percentile_dict(ticker,time_start,window,bands)
        path_containment.append(percentile_dict)

    print('Done', dt.datetime.today()-tic)
    ticker_standardized_returns = pd.concat(ticker_standardized_returns,axis=0)
    standardized_returns.append(ticker_standardized_returns)

path_containment = pd.DataFrame(path_containment).set_index(['ticker','path_start'])
standardized_returns = pd.concat(standardized_returns,axis=1)

In [ ]:
plot_bands_vs_actual(window,drift,bands,ticker,paths)
print(f'Overall Path Containment Stats ({path_containment.shape[0]} trials):\n',
      path_containment.mean(),sep='')

In [ ]:
diag_df = qq_panel(my_picks,standardized_returns)

In [ ]:
diag_df

In [ ]:
# Test for lagged/clustered relationship with returns, anything below .01 suggests clustering
for ticker in my_picks:
    z = standardized_returns[(ticker,'standardized returns')].dropna()
    lb = acorr_ljungbox(z**2, lags=[5, 10, 20], return_df=True)
    print(ticker, lb['lb_pvalue'].values)

In [ ]:
month_starts = make_month_grid(close)
window_results = []
garch_path_containment = []
saved_garch_coefficients = []

for ticker in my_picks:
    
    tic = dt.datetime.today()
    print(ticker,tic,end=' - ')
    ticker_window_result = []
    garch_coefficients = None
    
    # Skip 2 years for garch coefficient fitting, also throw out 1 month for last upper bound
    for month_select in range(24,len(month_starts)-1):

        time_start = month_starts[month_select]
        prev_data, window = isolate_data(close,ticker,month_starts,month_select)

        # Fit garch coefficients once per year (look back = 2 years)
        if garch_coefficients == None or time_start.month == 1:
            garch_coefficients = fit_garch(close,ticker,month_starts,month_select)
            garch_coefficients['ticker'] = ticker
            saved_garch_coefficients.append(garch_coefficients)

        # Simulations with GARCH-t paths
        ticker_id = my_picks.index(ticker)  # or any stable integer mapping
        time_start = month_starts[month_select]
        ss = np.random.SeedSequence([ticker_id, time_start.toordinal()])
        rng = np.random.default_rng(ss)
        drift, bands, paths, sigma2 = simulate_garch_paths(window, garch_coefficients, prev_data, rng)
        percentile_dict = make_percentile_dict(ticker,time_start,window,bands)
        garch_path_containment.append(percentile_dict)
        
        # Warm up garch on prev 3 months, then run on the selected month
        window_result = monthly_garch(garch_coefficients,prev_data,window)
        ticker_window_result.append(window_result)

        if garch_coefficients == None or time_start.month == 1:
            saved_garch_coefficients[-1]['final_trial_intial_sigma'] = window_result[(ticker,'garch sigma')].iloc[0]
            saved_garch_coefficients[-1]['final_trial_last_mean_path_sigma'] = window_result[(ticker,'garch sigma')].iloc[0]
            
    print('Done', dt.datetime.today()-tic)
    ticker_window_result = pd.concat(ticker_window_result)
    window_results.append(ticker_window_result)

window_results = pd.concat(window_results,axis=1)
garch_path_containment = pd.DataFrame(garch_path_containment).set_index(['ticker','path_start'])
saved_garch_coefficients = pd.DataFrame(saved_garch_coefficients)

In [ ]:
plot_bands_vs_actual(window,drift,bands,ticker,paths)
print(f'Overall Path Containment Stats ({path_containment.shape[0]} trials):\n',
      garch_path_containment.mean(),sep='')